# Checkpoint inference

You can use this notebook to load checkpoints/ours.pth, run inference on the prepared PI-CAI test slices, and display a prediction.

## Run in VS Code or Jupyter
1. Use Python 3.12 and select that environment as the notebook kernel.
2. Install the packages shown below, then restart the kernel.
3. Set ROOT to your MyGO folder and DATA_ROOT to the prepared data folder containing images/ and masks/0/.
4. Run all cells from top to bottom. CPU inference is used.
5. A missing checkpoint is downloaded from this repository.
6. Results are saved in local_results/picai_test_slices.csv and local_results/summary.json.

## Data requirements
This version has been tested on the prepared 256-by-256 PNG inputs for patients 11256 and 11382 (46 slices). The embedded manifest verifies their filenames and SHA-256 hashes. An example image filename is 11256_11256_1001279_t2w_0.png; its label is masks/0/11256_11256_1001279_0.png.

The public dataset at https://github.com/LZC0402/Dataset contains these patients in MR-images7.zip and their labels in label_lesions.zip. Those public PNGs have different dimensions and filenames. The exact conversion from that archive format to the verified inputs is pending, so provide the prepared data when running this version.

## Implementation
The matching model definitions are included below. They preserve the historical implementation, whose executed attention path uses dense self-attention. The historical training script used this evaluation set for epoch-wise model selection.

The network builds on U-KAN (https://github.com/CUHK-AIM-Group/U-KAN) and KAN layers. The definitions below are the checkpoint-matching version supplied with this model.


In [ ]:
# Run once in the selected environment, then restart the kernel:
# %pip install torch==2.5.1 torchvision==0.20.1
# %pip install timm==1.0.12 numpy==2.5.4 opencv-python-headless matplotlib==3.11.2
# %pip install ipykernel


In [ ]:
from pathlib import Path
import os, json, hashlib, csv, urllib.request, zipfile
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
ROOT = Path(os.environ.get("MYGO_ROOT", Path.cwd())).resolve()
DATA_ROOT = Path(os.environ.get("MYGO_DATA", ROOT / "data/mri")).resolve()
DOWNLOAD_MISSING = True
torch.set_num_threads(4)
torch.manual_seed(1029)
print("Working directory:", ROOT)
print("Data directory:", DATA_ROOT)
print("PyTorch", torch.__version__, "NumPy", np.__version__, "OpenCV", cv2.__version__)


In [ ]:
manifest = {'sha256': '25632833cfca0605b0973f2a246f2eee3c28125fb81e6ac518b535f599eb807a', 'test_files': {'11256_11256_1001279_t2w_0': {'image': '1a314ec0e0200b1412395b0d18ad95ba6aecfc87d79439f8ca03435a628066d3', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_1': {'image': 'a7dc838ed0be6bd1d7249d06852ee72c06bda28d352ee7c64e199f2fce9ccd0b', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_10': {'image': 'bf2ac073711167faacde439029715e30b0f38318d8f821aa4d3a078a9e4eafec', 'mask': 'e283a62ac5451ec87f3fdac752ed3faebeb1e2ea0b88684df22d619e493ca2dd'}, '11256_11256_1001279_t2w_11': {'image': '78b1b2b702d602b0f0c029185e490bb317d3964265ef56b3d1ca7d57ee82c28b', 'mask': '533d83582af73a79901d8c9f2659678b4109118d04f70d9680ef7b14f61ec8e1'}, '11256_11256_1001279_t2w_12': {'image': '3980161dfa01409e9e3b9907bdd1abbbfdc1f7d7a5915e14006db14923a80d16', 'mask': '14cefe17d785820d8edcdc8f9e2a99e5f3a532280405fe07d94df617819b7c84'}, '11256_11256_1001279_t2w_13': {'image': '929453c7c3bbfa86f17d87de9a8b142ff6931560986ba5a29231bfefffb212c6', 'mask': 'c4fb4464d3409d609a88d0466aec51561c6b9aaa7798bd1decc52ff033d87dee'}, '11256_11256_1001279_t2w_14': {'image': '8461e6498945b0117757b1952172488e9a57e69701210c7a641575f60339af18', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_15': {'image': '3e8564e68b732a4b3009bc17c0b3fffac6066d13529ae492bf4452af8e8a26e7', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_16': {'image': '825ba8bf365cdb456a46c4b41b818c8129007bbb650d19386eb7d3c5af735f89', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_17': {'image': '49e8f4de0005b6766c07ee5d9f90c69c86dca76d6dd882f5022f9e7a52d2d4e8', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_18': {'image': '4c96e98b228f236fc66183957ba348da606f346abee72c40bb4254e54114aaff', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_2': {'image': '430d441aad3cb35c94b2c328cb3869be837ed1cc29aabd1d55e0d43cbab9c52c', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_3': {'image': 'd42226a43c352e9f3b02bb8c4b21d9180887610e9534c99a450386fd0cbd8567', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_4': {'image': 'a3cdfd95fded4b0009c3e0fe409d16b33da263047bb12618c3f0d2bb32a0d25e', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11256_11256_1001279_t2w_5': {'image': '199d7409141846f54284034fb7296ba6284cb826fbfd11782ae8d2497ad1d47e', 'mask': 'ef86d52f5126901ad748da12bd0497818b0c01c4ddc2d031c1c0ac126e39ef73'}, '11256_11256_1001279_t2w_6': {'image': 'c85d9754dd4a88cd13ca95f5501b152175caf24e8aac5329930620ec2e923591', 'mask': '2de2a96f03d21233c3f12b82806c9cb838354ceb34bd58dcb0358a3f03aa8b97'}, '11256_11256_1001279_t2w_7': {'image': 'bb83f0da03a2a24c5bc8fcd077941ab80554be66f8ce3f3d6373ab6df6c8a44a', 'mask': '98faa9b333dff52c4cf21b38f0c3af81ecd98bb01bf2330229aa561f0398ea2f'}, '11256_11256_1001279_t2w_8': {'image': 'e3c242b0bcd250d3e11d4de569578baca8985f13bdeb4d429c4e19fe1cd4d029', 'mask': 'c42bee6a11e1e0e8a08eb907a43077b009cfb20c0b9ccd1b3b65cf494bc173a7'}, '11256_11256_1001279_t2w_9': {'image': 'ae41fc6e7427ec6d22f0309cfbbacdc37880229e223e0223d3fa2d213b9eb0ac', 'mask': '1c8a651b77ae5df22ef1e56ba0eec5894b56c6296c6f9181cb7f4847cbb8b369'}, '11382_11382_1001405_t2w_0': {'image': '0f03b2d8a8c47898968c7f6cbfcc870d1fa0870b2b82f02a8564b2cff03a1eef', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_1': {'image': '19f9d1c908ff2955f922774deb14b69cdbcf9488173eab252aa29f39fef7ceb4', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_10': {'image': '164cc905be1b3353ab800cf91da5e400f71fa7dfea7b0343c64fbc5c3bdaf8f2', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_11': {'image': '1cea80709f9429fb070c03c737c2f670bf9b135956ecbcf8cd5e7022334b6512', 'mask': 'afd0bf548533b65c073e6cadbd353e7df61b91583e489e184f73131f3d890474'}, '11382_11382_1001405_t2w_12': {'image': 'e14bef442ca318cb07e9c75e564832c48dd08ef3448acfa8f21f75025e6f97ea', 'mask': '8d0771d4b651f888f4f262acd8d9735730de6cfede112e23152bf37948dd7236'}, '11382_11382_1001405_t2w_13': {'image': 'c44237509402d8892a1e8a54b2abad61b7f78cf8b7101772509f3e31bdf759fe', 'mask': '5d3a32e79e0716c726eee203084e01edf04669855135413e5131ab88f9e11b6f'}, '11382_11382_1001405_t2w_14': {'image': 'b108ea35473493a857bc7f8aabc7d0eaf1effee2f45a3ae2d11c1e5a3a4acbc5', 'mask': '5d3a32e79e0716c726eee203084e01edf04669855135413e5131ab88f9e11b6f'}, '11382_11382_1001405_t2w_15': {'image': '9c8693b0e7d1a3d4e8768d067132b56bd2a397d3e30bbbc80c46ccb9848adfcb', 'mask': '233c7638d5c95affca5816697a1cad9951e5f7e9408c887efeaba9bba6db8011'}, '11382_11382_1001405_t2w_16': {'image': '28b430636e54404eac7dbcb132843d033744bdf7c531fb48651ba8a4106b7aab', 'mask': '87c6d27fadb3d03dce8c9e69e9748ddfa5921a0571a4586fc82fa0ded8b3dc1a'}, '11382_11382_1001405_t2w_17': {'image': '57149a1dd58147afb799465941a60479f93db221672f9dac8c78844514f651e9', 'mask': '39a6674e7874a1c8895f066258a02115c1fcba5cf9fd018bada5a1edbf12ef16'}, '11382_11382_1001405_t2w_18': {'image': 'cb4d1dbf2cc7c8a5b24a65cdba51d9c2085ae91823abe92c69b08cafd5ddbf44', 'mask': '39a6674e7874a1c8895f066258a02115c1fcba5cf9fd018bada5a1edbf12ef16'}, '11382_11382_1001405_t2w_19': {'image': 'de9a0621a6ac26f9c8170128b1b506bfc2f2d3358bc9d8e4c978bdefb0400e87', 'mask': '07d6d062957fbe48c6f96f419ed3d9fc41c752fb60dbd2ca651cde84ef15196a'}, '11382_11382_1001405_t2w_2': {'image': 'a552aa90aba9723ab0e8d8b932d19f0dce4578e676e21c8ff7a341160623b557', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_20': {'image': 'ed860bf74c9c120cc690c591002c3e03f780a29c4ee0c248217213a3ef33815d', 'mask': 'dbceaeb181960db7ef2607291c96f862644b145ce1faae9a49f4cb96bc5adfc4'}, '11382_11382_1001405_t2w_21': {'image': 'abcc738f59b2bff5bb5bf08a5ad3b668c5572e51e46654da2c47d08b95675ed3', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_22': {'image': 'e4a82fadc9c949ff73a03daaff5cf8da28d0346b95db165ae5b13eff47d68477', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_23': {'image': 'b793da5db11dae571ea1d40028b60592c438636896f5306be1ed5ad4ec7ad8c3', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_24': {'image': '683d736979bdcaa776bf99d038be4f7941a36a1df042595ef9d7707efa17b352', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_25': {'image': '7d6121d7ad6857c3ec16a1d6cb40e591b6688040b03f1384fdcbc96ec6e35821', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_26': {'image': 'd0b7b6a3ae13b8f03a3f7f89e1a50a706fa71236b9a54952251982eb384a0c13', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_3': {'image': 'e0eefeb2cf978cd461dafe87cdd9c07aa8f5ec6b85c1fe908fba21d5ac0f7fed', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_4': {'image': '4ca1f74f7b3317ed4ac70709e47e9cdf33ea78446c2b10886090f18c56249e5d', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_5': {'image': '967b5d5dceabc3c4f1de98662b8aefae383c938d4c2f898b831337eb64591f06', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_6': {'image': 'ab48da29cb36ee388690c98dc49ee73b7fc6d37d1a2225db3608e06fa983169e', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_7': {'image': '270ce011731cc4e335187794042b6f94935b0488da836af2e76d1bf36e707ba8', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_8': {'image': '4588e5dd4c538d98f0e417e2f35c71747c4f43cc5488d2caf3590962bbb5a226', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}, '11382_11382_1001405_t2w_9': {'image': 'b6f995d312c52142937fe5810218eb1c1f8ee908282f5d97f40f3b807829bd54', 'mask': '004c0da60be05efebf543eb7ed2a51aba399a57d19c58d74290344ce5c54a1f3'}}}


In [ ]:
def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    request = urllib.request.Request(url, headers={"User-Agent":"MyGO-inference"})
    with urllib.request.urlopen(request, timeout=120) as response, temporary.open("wb") as f:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
    temporary.replace(path)

checkpoint = ROOT / "checkpoints/ours.pth"
if not checkpoint.exists() and DOWNLOAD_MISSING:
    download("https://raw.githubusercontent.com/LZC0402/MyGO/main/checkpoints/ours.pth", checkpoint)
assert checkpoint.exists(), "Download checkpoints/ours.pth or enable DOWNLOAD_MISSING."
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == manifest["sha256"], "Checkpoint hash mismatch."
image_dir = DATA_ROOT / "images"
mask_dir = DATA_ROOT / "masks/0"
expected = {}
for name, hashes in manifest["test_files"].items():
    expected[("image",name+".png")] = (image_dir/(name+".png"), hashes["image"])
    mask_name = name.replace("_t2w","")+".png"
    expected[("mask",mask_name)] = (mask_dir/mask_name, hashes["mask"])

for (kind,name),(path,digest) in expected.items():
    assert path.exists(), f"Missing {path}. Set DATA_ROOT to the prepared 256-by-256 PNG dataset described above."
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Input hash mismatch: {path}"
names = sorted(manifest["test_files"])
assert len(names) == 46
print("Checkpoint and all 46 image/mask pairs verified.")


## Model definitions
Run this cell before loading the checkpoint. It includes the KAN layers, attention blocks, and encoder-decoder required by the saved weights.

In [ ]:
import argparse
import torch.nn as nn

class qkv_transform(nn.Conv1d):
    """Conv1d for qkv_transform"""

def str2bool(v):
    if v.lower() in ['true', 1]:
        return True
    elif v.lower() in ['false', 0]:
        return False
    else:
        raise argparse.ArgumentTypeError('Boolean value expected.')


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

import torch
import torch.nn.functional as F
import math


class KANLinear(torch.nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KANLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (
                torch.arange(-spline_order, grid_size + spline_order + 1) * h
                + grid_range[0]
            )
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)

        self.base_weight = torch.nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = torch.nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = torch.nn.Parameter(
                torch.Tensor(out_features, in_features)
            )

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (
                    torch.rand(self.grid_size + 1, self.in_features, self.out_features)
                    - 1 / 2
                )
                * self.scale_noise
                / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order : -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                # torch.nn.init.constant_(self.spline_scaler, self.scale_spline)
                torch.nn.init.kaiming_uniform_(self.spline_scaler, a=math.sqrt(5) * self.scale_spline)

    def b_splines(self, x: torch.Tensor):
        """
        Compute the B-spline bases for the given input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).

        Returns:
            torch.Tensor: B-spline bases tensor of shape (batch_size, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features

        grid: torch.Tensor = (
            self.grid
        )  # (in_features, grid_size + 2 * spline_order + 1)
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, : -(k + 1)])
                / (grid[:, k:-1] - grid[:, : -(k + 1)])
                * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1 :] - x)
                / (grid[:, k + 1 :] - grid[:, 1:(-k)])
                * bases[:, :, 1:]
            )

        assert bases.size() == (
            x.size(0),
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return bases.contiguous()

    def curve2coeff(self, x: torch.Tensor, y: torch.Tensor):
        """
        Compute the coefficients of the curve that interpolates the given points.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_features).
            y (torch.Tensor): Output tensor of shape (batch_size, in_features, out_features).

        Returns:
            torch.Tensor: Coefficients tensor of shape (out_features, in_features, grid_size + spline_order).
        """
        assert x.dim() == 2 and x.size(1) == self.in_features
        assert y.size() == (x.size(0), self.in_features, self.out_features)

        A = self.b_splines(x).transpose(
            0, 1
        )  # (in_features, batch_size, grid_size + spline_order)
        B = y.transpose(0, 1)  # (in_features, batch_size, out_features)
        solution = torch.linalg.lstsq(
            A, B
        ).solution  # (in_features, grid_size + spline_order, out_features)
        result = solution.permute(
            2, 0, 1
        )  # (out_features, in_features, grid_size + spline_order)

        assert result.size() == (
            self.out_features,
            self.in_features,
            self.grid_size + self.spline_order,
        )
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        return self.spline_weight * (
            self.spline_scaler.unsqueeze(-1)
            if self.enable_standalone_scale_spline
            else 1.0
        )

    def forward(self, x: torch.Tensor):
        assert x.dim() == 2 and x.size(1) == self.in_features

        base_output = F.linear(self.base_activation(x), self.base_weight)
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        return base_output + spline_output

    @torch.no_grad()
    def update_grid(self, x: torch.Tensor, margin=0.01):
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x)  # (batch, in, coeff)
        splines = splines.permute(1, 0, 2)  # (in, batch, coeff)
        orig_coeff = self.scaled_spline_weight  # (out, in, coeff)
        orig_coeff = orig_coeff.permute(1, 2, 0)  # (in, coeff, out)
        unreduced_spline_output = torch.bmm(splines, orig_coeff)  # (in, batch, out)
        unreduced_spline_output = unreduced_spline_output.permute(
            1, 0, 2
        )  # (batch, in, out)

        # sort each channel individually to collect data distribution
        x_sorted = torch.sort(x, dim=0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(
                0, batch - 1, self.grid_size + 1, dtype=torch.int64, device=x.device
            )
        ]

        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(
                self.grid_size + 1, dtype=torch.float32, device=x.device
            ).unsqueeze(1)
            * uniform_step
            + x_sorted[0]
            - margin
        )

        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.concatenate(
            [
                grid[:1]
                - uniform_step
                * torch.arange(self.spline_order, 0, -1, device=x.device).unsqueeze(1),
                grid,
                grid[-1:]
                + uniform_step
                * torch.arange(1, self.spline_order + 1, device=x.device).unsqueeze(1),
            ],
            dim=0,
        )

        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """
        Compute the regularization loss.

        This is a dumb simulation of the original L1 regularization as stated in the
        paper, since the original one requires computing absolutes and entropy from the
        expanded (batch, in_features, out_features) intermediate tensor, which is hidden
        behind the F.linear function if we want an memory efficient implementation.

        The L1 regularization is now computed as mean absolute value of the spline
        weights. The authors implementation also includes this term in addition to the
        sample-based regularization.
        """
        l1_fake = self.spline_weight.abs().mean(-1)
        regularization_loss_activation = l1_fake.sum()
        p = l1_fake / regularization_loss_activation
        regularization_loss_entropy = -torch.sum(p * p.log())
        return (
            regularize_activation * regularization_loss_activation
            + regularize_entropy * regularization_loss_entropy
        )


class KAN(torch.nn.Module):
    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        base_activation=torch.nn.SiLU,
        grid_eps=0.02,
        grid_range=[-1, 1],
    ):
        super(KAN, self).__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order

        self.layers = torch.nn.ModuleList()
        for in_features, out_features in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_features,
                    out_features,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    scale_noise=scale_noise,
                    scale_base=scale_base,
                    scale_spline=scale_spline,
                    base_activation=base_activation,
                    grid_eps=grid_eps,
                    grid_range=grid_range,
                )
            )

    def forward(self, x: torch.Tensor, update_grid=False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )

import torch
from torch import nn
import torch
import torchvision
from torch import nn
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import save_image
import torch.nn.functional as F
import os
import matplotlib.pyplot as plt


import timm
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
import types
import math
from abc import ABCMeta, abstractmethod
# from mmcv.cnn import ConvModule
from pdb import set_trace as st


from torch.nn import init

__all__ = ['UKAN', 'D_ConvLayer', 'ConvLayer', 'PatchEmbed', 'DW_bn_relu', 'DWConv', 'KANBlock', 'KANLayer']
class KANLayer(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0., no_kan=False):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.dim = in_features
        
        grid_size=5
        spline_order=3
        scale_noise=0.1
        scale_base=1.0
        scale_spline=1.0
        base_activation=torch.nn.SiLU
        grid_eps=0.02
        grid_range=[-1, 1]

        if not no_kan:
            self.fc1 = KANLinear(
                        in_features,
                        hidden_features,
                        grid_size=grid_size,
                        spline_order=spline_order,
                        scale_noise=scale_noise,
                        scale_base=scale_base,
                        scale_spline=scale_spline,
                        base_activation=base_activation,
                        grid_eps=grid_eps,
                        grid_range=grid_range,
                    )
            self.fc2 = KANLinear(
                        hidden_features,
                        out_features,
                        grid_size=grid_size,
                        spline_order=spline_order,
                        scale_noise=scale_noise,
                        scale_base=scale_base,
                        scale_spline=scale_spline,
                        base_activation=base_activation,
                        grid_eps=grid_eps,
                        grid_range=grid_range,
                    )
            self.fc3 = KANLinear(
                        hidden_features,
                        out_features,
                        grid_size=grid_size,
                        spline_order=spline_order,
                        scale_noise=scale_noise,
                        scale_base=scale_base,
                        scale_spline=scale_spline,
                        base_activation=base_activation,
                        grid_eps=grid_eps,
                        grid_range=grid_range,
                    )
            # # TODO   
            # self.fc4 = KANLinear(
            #             hidden_features,
            #             out_features,
            #             grid_size=grid_size,
            #             spline_order=spline_order,
            #             scale_noise=scale_noise,
            #             scale_base=scale_base,
            #             scale_spline=scale_spline,
            #             base_activation=base_activation,
            #             grid_eps=grid_eps,
            #             grid_range=grid_range,
            #         )   

        else:
            self.fc1 = nn.Linear(in_features, hidden_features)
            self.fc2 = nn.Linear(hidden_features, out_features)
            self.fc3 = nn.Linear(hidden_features, out_features)

        # TODO
        # self.fc1 = nn.Linear(in_features, hidden_features)


        self.dwconv_1 = DW_bn_relu(hidden_features)
        self.dwconv_2 = DW_bn_relu(hidden_features)
        self.dwconv_3 = DW_bn_relu(hidden_features)

        # # TODO
        # self.dwconv_4 = DW_bn_relu(hidden_features)
    
        self.drop = nn.Dropout(drop)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()
    

    def forward(self, x, H, W):
        # pdb.set_trace()
        B, N, C = x.shape

        x = self.fc1(x.reshape(B*N,C))
        x = x.reshape(B,N,C).contiguous()
        x = self.dwconv_1(x, H, W)
        x = self.fc2(x.reshape(B*N,C))
        x = x.reshape(B,N,C).contiguous()
        x = self.dwconv_2(x, H, W)
        x = self.fc3(x.reshape(B*N,C))
        x = x.reshape(B,N,C).contiguous()
        x = self.dwconv_3(x, H, W)

        # # TODO
        # x = x.reshape(B,N,C).contiguous()
        # x = self.dwconv_4(x, H, W)
    
        return x
class CrossAttention(nn.Module):
    def __init__(self, dim, out_dim, num_heads=1, qkv_bias=False, qk_scale=None, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        self.dim = dim
        self.out_dim = out_dim
        head_dim = out_dim // num_heads
        # NOTE scale factor was wrong in my original version, can set manually to be compat with prev weights
        self.scale = qk_scale or head_dim ** -0.5

        self.q_map = nn.Linear(dim, out_dim, bias=qkv_bias)
        self.k_map = nn.Linear(dim, out_dim, bias=qkv_bias)
        self.v_map = nn.Linear(dim, out_dim, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)

        self.proj = nn.Linear(out_dim, out_dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, q, v):
        B, N, _ = q.shape
        C = self.out_dim
        k = v
        NK = k.size(1)

        q = self.q_map(q).view(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        k = self.k_map(k).view(B, NK, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)
        v = self.v_map(v).view(B, NK, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x
class KANBlock(nn.Module):
    def __init__(self, dim, drop=0., drop_path=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm, no_kan=False):
        super().__init__()

        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim)

        self.layer = KANLayer(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop, no_kan=no_kan)
        self.cross=CrossAttention(mlp_hidden_dim,mlp_hidden_dim)
        self.self_attn=CrossAttention(mlp_hidden_dim,mlp_hidden_dim)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x, H, W):
        #print(self.drop_path(self.layer(self.norm2(x), H, W)).shape)
        x2=self.self_attn(x,x)
        #x2=self.cross(x,center)
        x1=self.drop_path(self.layer(self.norm2(x2), H, W))
        
        
        x = x + x1

        return x


class DWConv(nn.Module):
    def __init__(self, dim=768):
        super(DWConv, self).__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, bias=True, groups=dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W)
        x = self.dwconv(x)
        x = x.flatten(2).transpose(1, 2)

        return x

class DW_bn_relu(nn.Module):
    def __init__(self, dim=768):
        super(DW_bn_relu, self).__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, bias=True, groups=dim)
        self.bn = nn.BatchNorm2d(dim)
        self.relu = nn.ReLU()

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W)
        x = self.dwconv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = x.flatten(2).transpose(1, 2)

        return x

class PatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """

    def __init__(self, img_size=224, patch_size=7, stride=4, in_chans=3, embed_dim=768):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)

        self.img_size = img_size
        self.patch_size = patch_size
        self.H, self.W = img_size[0] // patch_size[0], img_size[1] // patch_size[1]
        self.num_patches = self.H * self.W
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride,
                              padding=(patch_size[0] // 2, patch_size[1] // 2))
        self.norm = nn.LayerNorm(embed_dim)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)

        return x, H, W


class ConvLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(ConvLayer, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, input):
        return self.conv(input)

class D_ConvLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(D_ConvLayer, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 3, padding=1),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, input):
        return self.conv(input)



class UKAN(nn.Module):
    def __init__(self, num_classes, input_channels=3, deep_supervision=False, img_size=224, patch_size=16, in_chans=3, embed_dims=[256, 320, 512], no_kan=False,
    drop_rate=0., drop_path_rate=0., norm_layer=nn.LayerNorm, depths=[1, 1, 1], **kwargs):
        super().__init__()

        kan_input_dim = embed_dims[0]

        self.encoder1 = ConvLayer(3, kan_input_dim//8)  
        self.encoder2 = ConvLayer(kan_input_dim//8, kan_input_dim//4)  
        self.encoder3 = ConvLayer(kan_input_dim//4, kan_input_dim)

        self.norm3 = norm_layer(embed_dims[1])
        self.norm4 = norm_layer(embed_dims[2])

        self.dnorm3 = norm_layer(embed_dims[1])
        self.dnorm4 = norm_layer(embed_dims[0])

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

        self.block1 = nn.ModuleList([KANBlock(
            dim=embed_dims[1], 
            drop=drop_rate, drop_path=dpr[0], norm_layer=norm_layer
            )])

        self.block2 = nn.ModuleList([KANBlock(
            dim=embed_dims[2],
            drop=drop_rate, drop_path=dpr[1], norm_layer=norm_layer
            )])

        self.dblock1 = nn.ModuleList([KANBlock(
            dim=embed_dims[1], 
            drop=drop_rate, drop_path=dpr[0], norm_layer=norm_layer
            )])

        self.dblock2 = nn.ModuleList([KANBlock(
            dim=embed_dims[0], 
            drop=drop_rate, drop_path=dpr[1], norm_layer=norm_layer
            )])

        self.patch_embed3 = PatchEmbed(img_size=img_size // 4, patch_size=3, stride=2, in_chans=embed_dims[0], embed_dim=embed_dims[1])
        self.patch_embed4 = PatchEmbed(img_size=img_size // 8, patch_size=3, stride=2, in_chans=embed_dims[1], embed_dim=embed_dims[2])

        self.decoder1 = D_ConvLayer(embed_dims[2], embed_dims[1])  
        self.decoder2 = D_ConvLayer(embed_dims[1], embed_dims[0])  
        self.decoder3 = D_ConvLayer(embed_dims[0], embed_dims[0]//4) 
        self.decoder4 = D_ConvLayer(embed_dims[0]//4, embed_dims[0]//8)
        self.decoder5 = D_ConvLayer(embed_dims[0]//8, embed_dims[0]//8)

        self.final = nn.Conv2d(embed_dims[0]//8, num_classes, kernel_size=1)
        self.soft = nn.Softmax(dim =1)

    def forward(self, x):
        
        B = x.shape[0]
        ### Encoder
        ### Conv Stage

        ### Stage 1
        out = F.relu(F.max_pool2d(self.encoder1(x), 2, 2))
        t1 = out
        ### Stage 2
        out = F.relu(F.max_pool2d(self.encoder2(out), 2, 2))
        t2 = out
        ### Stage 3
        out = F.relu(F.max_pool2d(self.encoder3(out), 2, 2))
        t3 = out

        ### Tokenized KAN Stage
        ### Stage 4

        out, H, W = self.patch_embed3(out)
        for i, blk in enumerate(self.block1):
            out = blk(out, H, W)
        out = self.norm3(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        t4 = out

        ### Bottleneck

        out, H, W= self.patch_embed4(out)
        for i, blk in enumerate(self.block2):
            out = blk(out, H, W)
        out = self.norm4(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

        ### Stage 4
        out = F.relu(F.interpolate(self.decoder1(out), scale_factor=(2,2), mode ='bilinear'))

        out = torch.add(out, t4)
        _, _, H, W = out.shape
        out = out.flatten(2).transpose(1,2)
        for i, blk in enumerate(self.dblock1):
            out = blk(out, H, W)

        ### Stage 3
        out = self.dnorm3(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()
        out = F.relu(F.interpolate(self.decoder2(out),scale_factor=(2,2),mode ='bilinear'))
        out = torch.add(out,t3)
        _,_,H,W = out.shape
        out = out.flatten(2).transpose(1,2)
        
        for i, blk in enumerate(self.dblock2):
            out = blk(out, H, W)

        out = self.dnorm4(out)
        out = out.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

        out = F.relu(F.interpolate(self.decoder3(out),scale_factor=(2,2),mode ='bilinear'))
        out = torch.add(out,t2)
        out = F.relu(F.interpolate(self.decoder4(out),scale_factor=(2,2),mode ='bilinear'))
        out = torch.add(out,t1)
        out = F.relu(F.interpolate(self.decoder5(out),scale_factor=(2,2),mode ='bilinear'))

        return self.final(out)


In [ ]:
model = UKAN(1, 3, False, embed_dims=[128,160,256])
model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True), strict=True)
model.eval()
print("Strict checkpoint loading succeeded.")


## Inference
Preprocessing preserves the original channel order, normalization, and additional division by 255. IoU and Dice are computed per slice and averaged over all 46 slices; an empty prediction with an empty label receives IoU = Dice = 1. The foreground threshold is strictly greater than 0.5.

In [ ]:
def load_slice(name):
    image = cv2.imread(str(image_dir / (name + ".png")))
    mask = cv2.imread(str(mask_dir / (name.replace("_t2w","") + ".png")), cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (256,256), interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (256,256), interpolation=cv2.INTER_NEAREST).astype(np.float32) / 255
    if mask.max() < 1:
        mask[mask > 0] = 1
    # Match original A.Normalize() (BGR order preserved), then Dataset's extra /255.
    mean = np.array([.485,.456,.406],dtype=np.float32) * 255
    inverse_std = np.reciprocal(np.array([.229,.224,.225],dtype=np.float32) * 255)
    normalized = (image.astype(np.float32) - mean) * inverse_std / 255
    return image, torch.from_numpy(normalized.transpose(2,0,1).copy())[None], mask > .5

rows = []
with torch.inference_mode():
    for name in names:
        image, x, gt = load_slice(name)
        probability = model(x).sigmoid()[0,0].numpy()
        prediction = probability > .5
        tp = int((prediction & gt).sum())
        fp = int((prediction & ~gt).sum())
        fn = int((~prediction & gt).sum())
        tn = int((~prediction & ~gt).sum())
        rows.append(dict(slice=name, tp=tp, fp=fp, fn=fn, tn=tn,
                         iou=tp/(tp+fp+fn) if tp+fp+fn else 1.,
                         dice=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 1.,
                         specificity=tn/(tn+fp) if tn+fp else 0.))
summary = {key:float(np.mean([r[key] for r in rows])) for key in ["iou","dice","specificity"]}
print("46-slice arithmetic means:", {key:round(value*100,4) for key,value in summary.items()})
output = ROOT / "local_results"
output.mkdir(exist_ok=True)
with (output/"picai_test_slices.csv").open("w",newline="") as f:
    writer = csv.DictWriter(f,fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
(output/"summary.json").write_text(json.dumps(summary,indent=2))


## Example prediction
The first annotated slice in filename order is displayed below.

In [ ]:
name = next(row["slice"] for row in rows if row["tp"] + row["fn"] > 0)
image, x, gt = load_slice(name)
with torch.inference_mode():
    probability = model(x).sigmoid()[0,0].numpy()
fig, axes = plt.subplots(1,3,figsize=(9,3))
for ax in axes:
    ax.axis("off")
axes[0].imshow(image[:,:,::-1]); axes[0].set_title("T2W input")
axes[1].imshow(gt,cmap="gray"); axes[1].set_title("Ground truth")
axes[2].imshow(probability > .5,cmap="gray"); axes[2].set_title("Prediction")
plt.tight_layout()
plt.show()
